# Digital Cash Management Forecast: lab (data scientist)

Explore the data and try models here without touching production: nothing in this notebook changes the flow, the
saved models or the production model.

1. **EDA**: money in, money out and net per client and month, totals over time, seasonality, and what the models
   learn.
2. **Models to try**: several forecasters trained and tested side by side on the same test as champion / challenger
   (trained without the last 6 months before the cut-off, tested on them), next to the production model and the
   simple guess (the average of the last 3 months), with the decision champion / challenger would take for each.
   Each try is recorded in Experiment Tracking, experiment `cashflow_lab` (Lab → Experiment tracking); lab tries
   are never deployed.
3. **Put one in production.**

**Kernel:** the code env `cashflow_forecast`. If it is not in the kernel list, ask whoever manages it to enable its
Jupyter support; meanwhile, the recipe `try_forecaster` tries one forecaster without a notebook (set the project
variable `lab_forecaster` and build `lab_results`, Lab zone).

In the sandbox the data is simulated: there, the scores show that the code works, not which model is better.

In [ ]:
import calendar
import importlib
import logging
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="google")      # third-party notices on import
logging.getLogger("matplotlib").setLevel(logging.ERROR)
try:
    import catboost
    import mlflow
except ImportError:
    raise RuntimeError("Use the kernel of the code env cashflow_forecast (Kernel > Change kernel). If it is not in the "
                       "list, ask whoever manages that code env to enable its Jupyter support; meanwhile, set the "
                       "project variable lab_forecaster and build lab_results (Lab zone).")
import dataiku

warnings.filterwarnings("ignore", category=DeprecationWarning)    # third-party notices; after dataiku, which shows them
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cashflow
import models
import tracking

for module in (cashflow, models, tracking):     # pick up changes to the project library without a restart
    importlib.reload(module)
from cashflow import read_dataset
from models import CAT_FEATURES, CatBoostForecaster, lab_compare, model_matrix

pd.set_option("display.max_columns", 30)
rows = read_dataset("training_rows")                        # what the models learn from
flows = read_dataset("monthly_flows")                       # money in and out per client and month
print(f"training_rows: {len(rows):,} rows, {rows['client_no'].nunique()} clients, cut-off months "
      f"{rows['cutoff_month'].min():%Y-%m} to {rows['cutoff_month'].max():%Y-%m}")

## 1. EDA

The models predict one value per client and month: the net cash flow (money in minus money out), 1, 2 and 3 months
ahead. First the monthly money in, money out and net per client, by segment (**negative_months**: the share of
months where more money went out than came in).

In [ ]:
# Money in, money out and net cash flow per client and month, by segment
monthly = (flows.pivot_table(index=["client_no", "month"], columns="direction", values="amount", aggfunc="sum")
           .reindex(columns=["IN", "OUT"]).fillna(0.0).reset_index())
monthly["NET"] = monthly["IN"] - monthly["OUT"]
monthly = monthly.merge(rows.drop_duplicates("client_no")[["client_no", "segment", "sole_proprietor"]], on="client_no")
overview = monthly.groupby("segment").agg(
    clients=("client_no", "nunique"), months=("month", "size"), money_in=("IN", "mean"), money_out=("OUT", "mean"),
    net=("NET", "mean"), negative_months=("NET", lambda s: (s < 0).mean())).round({"money_in": 0, "money_out": 0,
                                                                                  "net": 0, "negative_months": 3})
display(overview)
print("Sole proprietors:", int(rows.drop_duplicates("client_no")["sole_proprietor"].astype(str).str.lower()
                               .eq("true").sum()), "of", rows["client_no"].nunique(), "clients")

Totals over time and seasonality. Ramadan and Eid move about 11 days earlier each year, so a calendar-month profile
blurs them.

In [ ]:
# Totals over all clients per month: money in, money out and net (million SAR)
totals = monthly.groupby("month")[["IN", "OUT", "NET"]].sum() / 1e6
ax = totals.plot(figsize=(12, 4), title="All clients per month (million SAR)")
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_xlabel("")
plt.show()

# Seasonality without the trend: each month's total divided by the average of the 12 months before it (1 = usual)
ratio = totals[["IN", "OUT"]] / totals[["IN", "OUT"]].rolling(12).mean().shift(1)
profile = ratio.groupby(ratio.index.month).mean()
profile.index = [calendar.month_abbr[m] for m in profile.index]
ax = profile.plot.bar(figsize=(10, 3.5), title="Seasonality: month / the 12 months before, all clients (1 = usual)")
ax.axhline(1, color="grey", linewidth=0.5)
plt.show()

What the models learn: `target` (the net cash flow) divided by the client's scale (its usual month), its spread by
segment, and how much each input moves with it.

In [ ]:
# What the models learn: the net cash flow divided by the client's scale (its usual month), overall and by segment
scaled = rows.assign(y=rows["target"] / rows["scale"])
display(scaled.groupby("segment")["y"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]).round(2))

# How much each input moves with it (Spearman correlation, inputs divided by the scale too)
inputs = ["last_amount", "avg_3m", "same_month_last_year", "client_in_avg_3m", "client_out_avg_3m"]
x = scaled[inputs].div(scaled["scale"], axis=0)
display(x.loc[:, x.nunique() > 1].corrwith(scaled["y"], method="spearman").round(2).to_frame("correlation"))

## 2. Models to try

A forecaster is a class with `name`, `params` (its settings), `inputs` (the model-row columns it reads), `fit(rows)`
and `predict(rows)`, which returns `p10`, `ev` and `p90` in SAR (details in `models.py`, project library). Three
examples, next to the reference CatBoost forecaster (`catboost`, the production recipe):

- `simple_guess`: no model, the average of the last 3 months, with a band from its own past errors
- `catboost_depth8`: the reference CatBoost forecaster with deeper trees
- `sklearn_boosting`: scikit-learn gradient boosting, one model per output (EV, P10, P90)

Change them or add your own, then run the comparison. CatBoost, scikit-learn, pandas and matplotlib are in the code
env; other packages need the admin.

In [ ]:
class SimpleGuess:
    """The simple guess as a forecaster: EV = average of the last 3 months; P10 and P90 = EV plus the errors it made
    in training at 10% and 90%, scaled by the client's scale."""
    name = "simple_guess"
    params = {"quantiles": "0.1,0.9"}
    inputs = ["avg_3m", "scale"]

    def fit(self, rows):
        self.band = ((rows["target"] - rows["avg_3m"]) / rows["scale"]).quantile([0.1, 0.9]).values

    def predict(self, rows):
        ev, scale = rows["avg_3m"].values, rows["scale"].values
        return pd.DataFrame({"p10": ev + self.band[0] * scale, "ev": ev, "p90": ev + self.band[1] * scale},
                            index=rows.index)


class CatBoostDepth8(CatBoostForecaster):
    """The reference CatBoost forecaster with deeper trees."""
    name = "catboost_depth8"
    params = {**CatBoostForecaster.params, "depth": 8}


class SklearnBoosting:
    """scikit-learn gradient boosting (HistGradientBoostingRegressor): one model for the EV, one for P10 and one for
    P90 (quantile loss), with the reference forecaster's inputs and scaling (amounts / the client's scale)."""
    name = "sklearn_boosting"
    params = {"max_iter": 400, "learning_rate": 0.05, "max_leaf_nodes": 31, "min_samples_leaf": 40, "random_state": 0}
    inputs = CatBoostForecaster.inputs

    def _matrix(self, rows):
        X = model_matrix(rows)
        for c in CAT_FEATURES:            # categories as numbers; one not seen in training counts as missing
            X[c] = X[c].map(self.codes[c]).astype(float)
        return X

    def fit(self, rows):
        from sklearn.ensemble import HistGradientBoostingRegressor
        self.codes = {c: {v: i for i, v in enumerate(values.value_counts().index[:250])}   # at most 250 per column
                      for c, values in model_matrix(rows)[CAT_FEATURES].items()}
        X, y = self._matrix(rows), rows["target"] / rows["scale"]
        is_category = [c in CAT_FEATURES for c in X.columns]
        losses = {"ev": {"loss": "squared_error"}, "p10": {"loss": "quantile", "quantile": 0.1},
                  "p90": {"loss": "quantile", "quantile": 0.9}}
        self.models = {k: HistGradientBoostingRegressor(categorical_features=is_category, **loss, **self.params).fit(X, y)
                       for k, loss in losses.items()}

    def predict(self, rows):
        X, scale = self._matrix(rows), rows["scale"].values
        return pd.DataFrame({k: m.predict(X) * scale for k, m in self.models.items()}, index=rows.index)

In [ ]:
# Train each on the same months, test it on the last 6, next to the production model. catboost is the production
# recipe trained again: it should match the production model's scores (a check that the test is the same).
RECORD = True       # record each try in Experiment Tracking (experiment cashflow_lab)
result = lab_compare([SimpleGuess(), CatBoostForecaster(), CatBoostDepth8(), SklearnBoosting()], record=RECORD)
print("Net cash flow, all months ahead. EV error: lower is better. P10-P90 coverage: aim 80%.")
display(result["board"])
print("EV error per months ahead (ALL: together):")
display(result["errors"])
print("P10-P90 coverage per months ahead:")
display(result["coverage"])

In [ ]:
errors = result["errors"].drop(index="ALL")
if "simple_guess" in errors:                         # the same EV as the baseline: shown once
    errors = errors.drop(columns="simple_guess_baseline")
ax = errors.plot.bar(figsize=(10, 4), title="EV error per months ahead (lower is better)")
ax.set_xlabel("months ahead")
plt.tight_layout()
plt.show()

## 3. Put one in production

1. Copy its class into `forecasters.py` (project library) with the imports it needs (for example
   `import numpy as np`, `import pandas as pd`, `from models import CAT_FEATURES, CatBoostForecaster, model_matrix`)
   and add it to `FORECASTERS`. The setup notebook never overwrites that file.
2. Set the project variable `forecaster` to its name.
3. Run the scenario **Retrain, test and promote**. Champion / challenger promotes it only if it beats the
   production model on the same 6 months; the next Nightly batch forecasts with it.
4. To stop training it, set `forecaster` back to `catboost`. To use an earlier model now, activate its version in
   the three saved models.